In [3]:
# Install required libraries
!pip install -q sentence-transformers faiss-cpu

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ==========================================================
# 1. Create the Knowledge Base
# ==========================================================

documents = [
    """
    Generative Artificial Intelligence is a branch of AI that creates
    new content such as text, images, audio, video and computer programs.
    """,

    """
    Large Language Models are transformer-based models trained on massive
    text datasets. They are used for text generation, summarization,
    translation, question answering and conversational AI.
    """,

    """
    Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
    """,

    """
    Vector databases store high-dimensional embeddings and perform
    similarity searches. Examples of vector databases include FAISS,
    ChromaDB, Pinecone, Weaviate and Milvus.
    """,

    """
    Prompt engineering is the process of designing clear instructions
    that guide a language model to produce accurate and useful responses.
    Common techniques include zero-shot, few-shot and role-based prompting.
    """,

    """
    Fine-tuning adapts a pretrained language model to a specific domain
    or task by training it further using a smaller domain-specific dataset.
    """
]

# ==========================================================
# 2. Load Embedding Model
# ==========================================================

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ==========================================================
# 3. Create Document Embeddings
# ==========================================================

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
).astype("float32")

# Normalize embeddings
faiss.normalize_L2(document_embeddings)

# ==========================================================
# 4. Create FAISS Vector Database
# ==========================================================

dimension = document_embeddings.shape[1]

vector_database = faiss.IndexFlatIP(dimension)

vector_database.add(document_embeddings)

# ==========================================================
# 5. Retrieval Function
# ==========================================================

def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    similarity_scores, document_indices = vector_database.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []

    for index, score in zip(document_indices[0], similarity_scores[0]):

        retrieved_documents.append({
            "document": documents[index].strip(),
            "score": float(score)
        })

    return retrieved_documents

# ==========================================================
# 6. Generate Answer
# ==========================================================

def generate_answer(retrieved_documents):

    if len(retrieved_documents) == 0:
        return "The answer is not available in the knowledge base."

    return retrieved_documents[0]["document"]

# ==========================================================
# 7. Execute RAG System
# ==========================================================

print("RETRIEVAL-AUGMENTED GENERATION SYSTEM")
print("=" * 55)

user_query = input("\nEnter your question: ")

retrieved_results = retrieve_documents(
    user_query,
    top_k=2
)

answer = generate_answer(retrieved_results)

# ==========================================================
# 8. Display Retrieved Documents
# ==========================================================

print("\nRETRIEVED DOCUMENTS")
print("-" * 55)

for i, item in enumerate(retrieved_results, start=1):

    print(f"\nDocument {i}:")
    print(item["document"])
    print(f"Similarity Score: {item['score']:.4f}")

# ==========================================================
# 9. Display Generated Answer
# ==========================================================

print("\nGENERATED ANSWER")
print("-" * 55)

print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RETRIEVAL-AUGMENTED GENERATION SYSTEM

Enter your question: What is Retrieval-Augmented Generation?

RETRIEVED DOCUMENTS
-------------------------------------------------------

Document 1:
Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
Similarity Score: 0.6933

Document 2:
Generative Artificial Intelligence is a branch of AI that creates
    new content such as text, images, audio, video and computer programs.
Similarity Score: 0.3435

GENERATED ANSWER
-------------------------------------------------------
Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
